# Week 26, Day 6: Exam debrief

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w026d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


In [ ]:
def count_in(x, target):
    if isinstance(x, list):
        return sum(count_in(item, target) for item in x)
    return 1 if x == target else 0

print(count_in([1, [2, 1, [1, 3]], [], 1], 1), count_in([[1, 2], [1, 2]], [1, 2]))

In [ ]:
from functools import cache

def fewest_coins(amount, coins):
    @cache
    def best(a):
        if a == 0:
            return 0
        options = [best(a - c) for c in coins if c <= a]
        options = [o for o in options if o >= 0]
        return 1 + min(options) if options else -1
    return best(amount)

print(fewest_coins(6, (1, 3, 4)), fewest_coins(399, (1, 2, 5, 10, 20, 50, 100, 200)), fewest_coins(400, (6, 9)))

In [ ]:
def int_sqrt(n):
    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if mid * mid <= n:
            lo = mid
        else:
            hi = mid - 1
    return lo

print(int_sqrt(99), int_sqrt(100), int_sqrt(10**40))

In [ ]:
import re

def natural_key(name):
    parts = re.split(r"(\d+)", name.lower())
    return [int(p) if p.isdigit() else p for p in parts]

print(natural_key("photo10.jpg"))
print(sorted(["photo10.jpg", "photo2.jpg", "Photo1.jpg", "photo1b.jpg"], key=natural_key))

In [ ]:
import json
from datetime import datetime

def minutes_by_user(text):
    totals = {}
    for s in json.loads(text):
        start = datetime.fromisoformat(s["start"])
        end = datetime.fromisoformat(s["end"])
        if end < start:
            raise ValueError(f"session for {s['user']} ends before it starts")
        totals[s["user"]] = totals.get(s["user"], 0) + int((end - start).total_seconds() // 60)
    return dict(sorted(totals.items()))

print(minutes_by_user('[{"user": "bo", "start": "2026-11-02T23:30:00-05:00", "end": "2026-11-03T05:00:00Z"}]'))

In [ ]:
def top_customers(orders):
    totals = {}
    for customer, pence in orders:
        totals[customer] = totals.get(customer, 0) + pence    # 1: add, don't replace
    ranked = sorted(totals.items(), key=lambda p: (-p[1], p[0]))   # 2: biggest first, A to Z
    return ranked[:3]                                         # 3: three, not two

print(top_customers([("Cy", 500), ("Ade", 300), ("Bo", 700), ("Ade", 400), ("Di", 700), ("Ed", 100)]))

In [ ]:
def is_isbn10(text):
    s = text.replace("-", "")
    if len(s) != 10 or not s[:9].isdigit():
        return False
    if not (s[9].isdigit() or s[9] == "X"):
        return False
    values = [int(c) for c in s[:9]] + [10 if s[9] == "X" else int(s[9])]
    return sum(v * w for v, w in zip(values, range(10, 0, -1))) % 11 == 0

valid = ["0306406152", "0-306-40615-2", "080442957X", "0-8044-2957-X"]
invalid = ["0306406153", "080442957x", "0306406152X", "03064061", "0X06406155", ""]
print(all(is_isbn10(t) for t in valid), not any(is_isbn10(t) for t in invalid))

These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('timetable.csv', 'w', encoding='utf-8') as f:
    f.write('train,from,to,departs,arrives\nR1,Ashby,Brent,09:00,09:40\nR2,Brent,Calder,09:44,10:10\nR3,Brent,Calder,09:45,10:20\nR4,Ashby,Calder,09:05,10:30\nR5,Ashby,Brent,9:10,9:50\nR6,Brent,Calder,10:00,10:20\nR7,Ashby,Calder,10:00,10:25\nR8,Ashby,Dene,08:30,08:20\nR9,Calder,Ashby,1O:30,11:00\nR10,Ashby,Brent,09:00,09:40\n')

In [ ]:
import csv
import io

def _minutes(text):
    h, m = text.strip().split(":")
    if not (h.isdigit() and m.isdigit() and len(m) == 2 and 0 <= int(h) < 24 and 0 <= int(m) < 60):
        raise ValueError(f"bad time {text!r}")
    return int(h) * 60 + int(m)

def load(text):
    trains, problems = [], []
    reader = csv.DictReader(io.StringIO(text))
    for row in reader:
        try:
            departs, arrives = _minutes(row["departs"]), _minutes(row["arrives"])
            if arrives <= departs:
                raise ValueError("arrives before it departs")
        except ValueError:
            problems.append(f"line {reader.line_num}")
            continue
        trains.append({"train": row["train"].strip(), "from": row["from"].strip(), "to": row["to"].strip(),
                       "departs": departs, "arrives": arrives})
    return trains, problems

def departures(trains, station, after):
    chosen = [t for t in trains if t["from"] == station and t["departs"] >= after]
    return sorted(chosen, key=lambda t: (t["departs"], t["train"]))

def best_journey(trains, start, end, after):
    candidates = []
    for first in departures(trains, start, after):
        if first["to"] == end:
            candidates.append((first["arrives"], 1, -first["departs"], [first["train"]]))
        else:
            for second in departures(trains, first["to"], first["arrives"] + 5):
                if second["to"] == end:
                    candidates.append((second["arrives"], 2, -first["departs"], [first["train"], second["train"]]))
    if not candidates:
        return None
    arrival, _, _, names = min(candidates)
    return arrival, names

def plan(text, start, end, after):
    trains, _ = load(text)
    best = best_journey(trains, start, end, _minutes(after))
    if best is None:
        return "No journey"
    arrival, names = best
    return f"Take {' then '.join(names)}, arrive {arrival // 60:02d}:{arrival % 60:02d}"

text = open("timetable.csv").read()
trains, problems = load(text)
print(problems)
print([t["train"] for t in departures(trains, "Ashby", 540)])
print(best_journey(trains, "Ashby", "Calder", 540))
print(plan(text, "Ashby", "Calder", "09:00"))
print(plan(text, "Ashby", "Calder", "09:11"))
print(plan(text, "Calder", "Ashby", "06:00"))

### Exercise: Second chance: nested totals

A shopping list is nested in sections: each item is either a `(name, pence)` tuple or a list of further items. Write `total_cost(items)` recursively.

In [ ]:
def total_cost(items):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"values\"\n    shop = [(\"bread\", 120), [(\"milk\", 95), [(\"tea\", 250)], []], (\"eggs\", 210)]\n    assert total_cost(shop) == 675 and total_cost([]) == 0 and total_cost([[[]]]) == 0")

### Exercise: Second chance: climbing stairs

You can climb a staircase 1, 2 or 3 steps at a time. Write `climbs(n)`: how many different ways are there to climb `n` steps? (`climbs(4)` is 7.) It must handle `n` = 200 quickly.

In [ ]:
from functools import cache

def climbs(n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"values\"\n    assert [climbs(n) for n in range(7)] == [1, 1, 2, 4, 7, 13, 24]\n    assert climbs(200) == 52622583840983769603765180599790256716084480555530641")

### Exercise: Second chance: the first bad build

Builds are numbered 1 to `n`. Every build from some point on is broken, and `is_broken(k)` tells you whether build `k` is. Write `first_broken(n, is_broken)` returning the first broken build, calling `is_broken` at most about log₂(n) + 2 times. Every list of builds has at least one broken build: build `n` always is.

In [ ]:
def first_broken(n, is_broken):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"values, with few calls\"\n    import math\n    for n in (1, 2, 3, 10, 1000, 10**9):\n        for bad in sorted({1, n, max(1, n // 3), max(1, n - 1)}):\n            calls = []\n            def check(k):\n                calls.append(k)\n                return k >= bad\n            assert first_broken(n, check) == bad, (n, bad)\n            assert len(calls) <= math.log2(n) + 2, (n, bad, len(calls))")

### Exercise: Second chance: version numbers

Write `sort_versions(versions)` that sorts version strings like `"1.10.2"` numerically, part by part, so `"1.9"` comes before `"1.10"`. A shorter version that matches the start of a longer one comes first (`"2.1"` before `"2.1.0"`).

In [ ]:
def sort_versions(versions):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"values\"\n    assert sort_versions([\"1.10\", \"1.9\", \"1.9.1\", \"0.12\", \"2.1.0\", \"2.1\", \"10.0\"]) == [\"0.12\", \"1.9\", \"1.9.1\", \"1.10\", \"2.1\", \"2.1.0\", \"10.0\"]\n    assert sort_versions([]) == []")

### Exercise: Second chance: meeting times

Write `local_times(utc_text, zones)`: given a UTC time like `"2026-11-02T14:00:00Z"` and a list of time zone names, return a list of strings like `"Africa/Lagos 15:00 Mon"`, one per zone, in the same order.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

def local_times(utc_text, zones):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"values\"\n    assert local_times(\"2026-11-02T14:00:00Z\", [\"Africa/Lagos\", \"America/New_York\", \"Asia/Tokyo\"]) == [\n        \"Africa/Lagos 15:00 Mon\", \"America/New_York 09:00 Mon\", \"Asia/Tokyo 23:00 Mon\"]\n    assert local_times(\"2026-07-01T23:30:00Z\", [\"Europe/London\", \"Asia/Kolkata\"]) == [\"Europe/London 00:30 Thu\", \"Asia/Kolkata 05:00 Thu\"]")

### Exercise: Second chance: tests that bite

`clamp(x, lo, hi)` in `limits.py` returns `x` limited to the range `lo` to `hi`, and raises `ValueError` if `lo > hi`. Write tests that catch all four broken versions.

In [ ]:
# Run this cell first: it creates the files used below.
with open('limits.py', 'w', encoding='utf-8') as f:
    f.write('def clamp(x, lo, hi):\n    if lo > hi:\n        raise ValueError("lo must not be more than hi")\n    return max(lo, min(x, hi))\n')

In [ ]:
import pytest
from limits import clamp

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"limits.py\").read()\nBROKEN = {\n    \"swapped\": GOOD.replace(\"max(lo, min(x, hi))\", \"min(lo, max(x, hi))\"),\n    \"no check\": GOOD.replace('    if lo > hi:\\n        raise ValueError(\"lo must not be more than hi\")\\n', \"\"),\n    \"equal rejected\": GOOD.replace(\"lo > hi\", \"lo >= hi\"),\n    \"top ignored\": GOOD.replace(\"max(lo, min(x, hi))\", \"max(lo, x)\"),\n}\ndef test_catches():\n    \"your tests pass on the real code and catch all four broken versions\"\n    results = against(source, \"limits\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"limits.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the real code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Your score

Write down your total before reading on. Then, as you go through each problem below, note **why** you lost any marks. There are only a few kinds of reason, and knowing which kind you make most is more useful than the score itself:

1. **Didn't know it**: a gap in what you've learned. Revise the topic.
2. **Misread the question**: the code did something sensible but different. Slow down; underline every requirement.
3. **Edge case**: the main case worked, but empty input, a tie or a boundary failed. Test small and extreme cases.
4. **Ran out of time**: the plan, not the knowledge, was the problem. Practise starting with the parts you're sure of.

| Score | What it means |
|---|---|
| 144 or more (80%) | Distinction. You're well prepared for Phase 3. |
| 108 to 143 (60%) | Pass. Revise the topics where you lost marks during the rest week. |
| under 108 | Not yet. Spend the rest week on the topics below, then take the exam again. What matters is the second attempt. |

## Part A

| Questions | The idea | Revise |
|---|---|---|
| 1, 2, 3 | tracing recursion: what comes back up, and when code after the call runs; the recursion limit | [Week 15, Day 3](w015d3.html) |
| 4, 5 | memoisation; counting orderings and subsets | [Week 16, Day 3](w016d3.html), [Week 16, Day 4](w016d4.html) |
| 6, 7, 8 | Big-O of halving (or thirding) loops; scaling a quadratic; hidden loops and O(1) lookups | [Week 17](w017d3.html) |
| 9, 10, 11 | binary search steps; which sorts are O(n log n) at worst; stability, even with `reverse=True` | [Week 18](w018d2.html), [Week 19](w019d5.html) |
| 12, 13 | how import searches `sys.path`; `Counter` | [Week 20, Day 1](w020d1.html), [Week 20, Day 5](w020d5.html) |
| 14, 15 | merge conflicts; what not to commit | [Week 21, Day 4](w021d4.html), [Week 22, Day 2](w022d2.html) |
| 16, 17 | `re.sub` with `\s+`; `match` versus `fullmatch` | [Week 23, Day 1](w023d1.html) |
| 18, 19 | JSON's `null` and `true`; why CSV needs `newline=""` | [Week 23, Day 3](w023d3.html), [Week 23, Day 4](w023d4.html) |
| 20 | tests at boundaries | [Week 24, Day 3](w024d3.html) |

Question 11 catches many people: `reverse=True` doesn't reverse the order of equal items, because Python's sort stays stable in both directions.

## Part B

Each model solution runs, so try it. The notes under each one explain the key decision and the most common mistakes.

### B1: Count in nested lists

**The key idea:** the base case is "not a list", not "an empty list". Then an empty list needs no special case: the sum of nothing is 0.

**Common mistakes:** checking `x == target` before checking whether `x` is a list (the last test, where the target is itself a list, is there to catch that); and an accumulator passed down as a mutable default argument.

### B2: Fewest coins

**The key idea:** the best answer for `a` is one coin plus the best answer for what's left, tried for every coin. Memoisation makes it about `amount × len(coins)` steps instead of exponential.

**Common mistakes:** the **greedy** approach (take the biggest coin that fits, repeatedly) takes 4 + 1 + 1 for 6 with `(1, 3, 4)`: three coins, when 3 + 3 needs only two. Greedy works for real British coins, but not in general. Also: letting -1 ("impossible") win the `min`, and defining the cached function outside, so the cache mixes up different coin sets.

### B3: Predict

- `a`: the pairs `(x, y)` with `x < y` in `[3, 1, 2]` are (1, 3), (1, 2), (2, 3): **3**.
- `b`: two loops over the list: **O(n^2)**.
- `c`: 2 to the 6: **64**.
- `d`: capital letters sort before lower case, and text compares character by character, so `"b10"` comes before `"b9"`: **["B1", "b10", "b9"]**. B5 fixes exactly this.
- `e`: **False**. Tests can only show the bugs they look for.

### B4: Whole square roots

**The key idea:** binary search doesn't need a list, only a yes/no question whose answer flips once: "is `mid * mid <= n`?" is yes up to the answer and no after it. Searching for the **last yes** means `lo = mid` on yes, so `mid` must round **up** (`+ 1`), or `lo = 0, hi = 1` loops for ever. That's the same trap as Day 2's bug hunt, from the other side.

**Common mistakes:** `** 0.5` (banned here, and wrong for big numbers anyway, because floats only keep about 16 digits); and the endless loop above.

### B5: Sort like a person

**The key idea:** a key function turns each name into something that compares the way you want. `re.split` with a **group** keeps the separators, so the digit runs stay in the list, and they always land at odd positions: text, number, text, number... That's why the lists never compare a number with a string.

**Common mistakes:** `re.findall(r"\d+|\D+")`, which loses that alternation when a name starts with a digit, so `"9"` and `"b"` compare an int with a string and crash; forgetting `lower()`; and sorting by the numbers alone, which loses the text.

### B6: Sessions across time zones

**The key idea:** aware datetimes subtract correctly across time zones, so there's no need to convert anything by hand. 23:30 at UTC−5 is 04:30 UTC, so Bo's session is 30 minutes, though the clock times look 5½ hours apart and on different days.

**Common mistakes:** cutting the offset off and treating the times as local (that gives 330 minutes for Bo); `delta.seconds` instead of `total_seconds()` (`.seconds` ignores whole days and wraps round); and forgetting the alphabetical order.

### B7: Three bugs

**The key idea:** `reverse=True` reverses **every** part of a tuple key, so names came out Z to A. To sort one part down and another up, negate the number instead.

**Common mistakes:** fixing two bugs and stopping. The task said three: when you're told how many there are, keep looking until you've found them all, and test a case for each.

### B8: Test an ISBN checker

As pytest tests, these become two parametrized tests, one for each list.

**The key idea:** each invalid example breaks **exactly one** rule and would pass if that rule were missing. `"0X06406155"` is the subtle one: its checksum works if `X` is allowed to count as 10 anywhere, so only the "X only at the end" rule rejects it. A random string of junk fails several checks at once and so can't tell you whether any single one is there.

**Common mistakes:** only testing valid numbers; invalid examples that fail for several reasons; and not testing hyphens, the lower-case `x`, or a string that's too long.

## Part C

**The key ideas:**

- **Decomposition**: `best_journey` uses `departures` twice, once for the first train and once for the connection, so the rules about times and stations live in one place.
- **Generate, then choose**: collect every candidate journey, then pick the best with `min` and a tuple key, `(arrival, number of trains, -departure, names)`. Each tie-breaking rule from the question is one part of the tuple, in order. That's far easier to get right than a loop full of `if better than the best so far` comparisons.
- **The connection rule**: the second train must leave at least 5 minutes after the first arrives, so `departures(trains, first["to"], first["arrives"] + 5)`. R1 arrives at Brent at 09:40, and R2 leaves at 09:44: too tight. If you allowed it, you'd have arrived at 10:10 by R1 then R2, a journey that doesn't exist.
- **Clean at the edge**: `load` turns every time into minutes once. `"9:10"` is a perfectly good time (the hours have one digit); `"1O:30"` (a letter O) isn't.

**Common mistakes:** comparing times as text (`"9:50" > "10:20"` is True for strings); forgetting the arrives-after-departs check (R8); choosing the direct train because it's direct, before comparing arrival times; and the tie rules in the wrong order.

## Second chances

A fresh problem for each skill the exam tested. Try the ones for any problem where you lost marks.

## Quiz

## What's next

**Your revision plan.** For each problem where you lost marks, write one line: the problem, the kind of mistake (from the list at the top), and the day you'll revisit. Do those days during the rest week. If you're retaking the exam, do it after the rest week, without looking at this page again first.

**Phase 3: Objects and design.** After a week's rest, Phase 3 is about organising bigger programs. You'll learn to write your own classes, starting from why they exist at all, and work up to Python's data model, type hints, and the design principles and patterns that keep large programs understandable. You'll see spend again: its dictionaries of expenses are exactly the kind of thing a class does better.

You've finished Phase 2. Twelve weeks ago you met a method for solving problems; since then you've written recursive algorithms, measured and compared them, built real tools with modules, Git, data formats and tests, and finished a project from an empty file. Take the rest week: you've earned it.